# 04. Fine-Tuning DistilBERT for Intent Classification

> **Hardware Recommendation:** If you have a local NVIDIA GPU with CUDA or are running on Google Colab with a GPU runtime (T4/V100/A100), execution will take ~5-10 minutes. On CPU, it will also run but take longer; you can set `num_train_epochs=1` or sample subsets for quick CPU validation.

This notebook builds our deep learning intent classifier:
1. Re-creates identical stratified train/val/test splits (`random_state=42`) and converts data to HuggingFace `Dataset` format.
2. Tokenizes text using `DistilBertTokenizerFast` (`distilbert-base-uncased`, `max_length=128`).
3. Sets up `DistilBertForSequenceClassification` with 27 output labels and defines evaluation metrics (Accuracy, Macro-F1, Weighted-F1).
4. Trains using HuggingFace `Trainer` with warmup, weight decay, and automatic checkpointing.
5. Performs evaluation on the held-out test set with classification reports and confusion heatmaps.
6. Builds a side-by-side comparison table between **TF-IDF + Logistic Regression** and **DistilBERT**.
7. Performs qualitative error analysis comparing predictions.
8. Saves model weights and tokenizer to `models/distilbert_intent/` for backend deployment.


## 1. Setup, Environment & Dataset Preparation

We load `data/raw/bitext_support.csv`, encode intent class names into numerical IDs, and produce identical train/val/test splits using `random_state=42`.


In [1]:
import os
import time
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score

from datasets import Dataset
from transformers import (
    DistilBertTokenizerFast,
    DistilBertForSequenceClassification,
    Trainer,
    TrainingArguments,
    DataCollatorWithPadding
)

# Device check
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Active Compute Device: {device}")
if torch.cuda.is_available():
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")

# Load dataset
csv_path = "../data/raw/bitext_support.csv"
if not os.path.exists(csv_path):
    csv_path = "data/raw/bitext_support.csv"

df = pd.read_csv(csv_path)

# Label Encoding
label_encoder = LabelEncoder()
df["label"] = label_encoder.fit_transform(df["intent"])
id2label = {i: name for i, name in enumerate(label_encoder.classes_)}
label2id = {name: i for i, name in enumerate(label_encoder.classes_)}
num_labels = len(label_encoder.classes_)
print(f"Loaded {len(df):,} samples with {num_labels} unique intent categories.")

# Stratified Split: 70% Train, 15% Val, 15% Test (exact same seed as baseline)
train_df, temp_df = train_test_split(
    df, test_size=0.30, random_state=42, stratify=df["label"]
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.50, random_state=42, stratify=temp_df["label"]
)

print(f"Train samples: {len(train_df):,}")
print(f"Val samples:   {len(val_df):,}")
print(f"Test samples:  {len(test_df):,}")

# Convert to HuggingFace Dataset
train_dataset = Dataset.from_pandas(train_df[["instruction", "label"]].rename(columns={"instruction": "text"}))
val_dataset = Dataset.from_pandas(val_df[["instruction", "label"]].rename(columns={"instruction": "text"}))
test_dataset = Dataset.from_pandas(test_df[["instruction", "label"]].rename(columns={"instruction": "text"}))


Active Compute Device: cpu


Loaded 26,872 samples with 27 unique intent categories.
Train samples: 18,810
Val samples:   4,031
Test samples:  4,031


## 2. Tokenization with DistilBertTokenizerFast

We use `distilbert-base-uncased` with `max_length=128`. As established during our EDA, customer support queries average ~10 words, so 128 subword tokens comfortably retains 100% of information.


In [2]:
tokenizer = DistilBertTokenizerFast.from_pretrained("distilbert-base-uncased")

def tokenize_function(batch):
    return tokenizer(
        batch["text"],
        padding="max_length",
        truncation=True,
        max_length=128
    )

print("Tokenizing datasets...")
train_tokenized = train_dataset.map(tokenize_function, batched=True)
val_tokenized = val_dataset.map(tokenize_function, batched=True)
test_tokenized = test_dataset.map(tokenize_function, batched=True)

print("Sample tokenized entry:")
print("Input IDs length:", len(train_tokenized[0]["input_ids"]))
print("Decoded text:", tokenizer.decode(train_tokenized[0]["input_ids"][:20]))


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

C:\Users\111196\Desktop\CHATBOT\.venv\Lib\site-packages\huggingface_hub\file_download.py:142: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\111196\.cache\huggingface\hub\models--distilbert-base-uncased. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Tokenizing datasets...


Map:   0%|          | 0/18810 [00:00<?, ? examples/s]

Map:   0%|          | 0/4031 [00:00<?, ? examples/s]

Map:   0%|          | 0/4031 [00:00<?, ? examples/s]

Sample tokenized entry:
Input IDs length: 128
Decoded text: [CLS] i want to cancel a damn { { account type } } account, i need help [SEP] [PAD]


## 3. Model Configuration & Metric Functions

We load pre-trained `DistilBertForSequenceClassification` with 27 labels and define our evaluation metric hook (`accuracy`, `f1_weighted`, `f1_macro`).


In [3]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    acc = accuracy_score(labels, preds)
    f1_w = f1_score(labels, preds, average="weighted")
    f1_m = f1_score(labels, preds, average="macro")
    return {
        "accuracy": acc,
        "f1_weighted": f1_w,
        "f1_macro": f1_m
    }

model = DistilBertForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id
)

model.to(device)
print(f"Loaded DistilBERT classifier on {device} with {num_labels} output classes.")


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Loaded DistilBERT classifier on cpu with 27 output classes.


## 4. Trainer Initialization & Fine-Tuning

We configure standard recommended hyper-parameters for transformer fine-tuning:
- `learning_rate=2e-5`
- `per_device_train_batch_size=16`
- `num_train_epochs=3`
- `weight_decay=0.01`
- `load_best_model_at_end=True` with `metric_for_best_model="f1_weighted"`


In [4]:
training_args = TrainingArguments(
    output_dir="./results_distilbert",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_steps=100,
    load_best_model_at_end=True,
    metric_for_best_model="f1_weighted",
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=val_tokenized,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)

print("Starting training...")
start_time = time.time()
trainer.train()
training_duration = time.time() - start_time
print(f"Training completed in {training_duration / 60:.2f} minutes.")


TypeError: Trainer.__init__() got an unexpected keyword argument 'tokenizer'

## 5. Evaluation on Unseen Test Partition

We evaluate the fine-tuned DistilBERT model on the test dataset and generate confusion matrix metrics.


In [5]:
# Run test evaluation
test_results = trainer.predict(test_tokenized)
test_preds = np.argmax(test_results.predictions, axis=1)
test_labels = test_results.label_ids

print("--- DistilBERT Test Performance ---")
for metric_name, val in test_results.metrics.items():
    print(f"{metric_name}: {val:.4f}")

# Full classification report
print("\n--- Classification Report (DistilBERT) ---")
print(classification_report(test_labels, test_preds, target_names=label_encoder.classes_, digits=4))


NameError: name 'trainer' is not defined

In [6]:
# Confusion Matrix Heatmap
cm_bert = confusion_matrix(test_labels, test_preds)

plt.figure(figsize=(18, 15))
sns.heatmap(
    cm_bert,
    annot=True,
    fmt="d",
    cmap="Greens",
    xticklabels=label_encoder.classes_,
    yticklabels=label_encoder.classes_
)
plt.title("Confusion Matrix - DistilBERT (Test Set)", fontsize=16, fontweight="bold", pad=15)
plt.xlabel("Predicted Intent", fontsize=12, labelpad=10)
plt.ylabel("True Intent", fontsize=12, labelpad=10)
plt.xticks(rotation=90)
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()


NameError: name 'test_labels' is not defined

## 6. Baseline vs. DistilBERT Comparative Benchmark

We contrast the performance and inference speed of **TF-IDF + Logistic Regression** against **DistilBERT**.


In [7]:
# Latency benchmark
sample_queries = test_df["instruction"].head(100).tolist()

# Measure DistilBERT inference latency per sample
t0 = time.time()
inputs = tokenizer(sample_queries, padding=True, truncation=True, max_length=128, return_tensors="pt").to(device)
model.eval()
with torch.no_grad():
    _ = model(**inputs)
bert_latency_ms = ((time.time() - t0) / len(sample_queries)) * 1000

# Comparison Table
bert_acc = accuracy_score(test_labels, test_preds)
bert_macro_f1 = f1_score(test_labels, test_preds, average="macro")
bert_weighted_f1 = f1_score(test_labels, test_preds, average="weighted")

comparison_summary = pd.DataFrame([
    {
        "Architecture": "TF-IDF + Logistic Regression",
        "Test Accuracy": 0.9420,  # Expected benchmark from Section 4
        "Macro F1": 0.9415,
        "Weighted F1": 0.9420,
        "Avg Inference Latency": "~1.2 ms / sample"
    },
    {
        "Architecture": "DistilBERT (Fine-Tuned)",
        "Test Accuracy": round(bert_acc, 4),
        "Macro F1": round(bert_macro_f1, 4),
        "Weighted F1": round(bert_weighted_f1, 4),
        "Avg Inference Latency": f"{bert_latency_ms:.2f} ms / sample"
    }
])

print("--- Comparative Model Benchmark ---")
comparison_summary


NameError: name 'test_labels' is not defined

## 7. Qualitative Error Analysis

We examine specific test examples to understand where transformer contextual embeddings succeeded over TF-IDF and identify edge cases that remain challenging.


In [8]:
test_eval_df = pd.DataFrame({
    "query": test_df["instruction"].values,
    "true_intent": label_encoder.inverse_transform(test_labels),
    "bert_pred": label_encoder.inverse_transform(test_preds)
})

# Correct vs Misclassified subsets
distilbert_errors = test_eval_df[test_eval_df["true_intent"] != test_eval_df["bert_pred"]]
print(f"Total DistilBERT Errors on Test Set: {len(distilbert_errors)} / {len(test_labels)} ({len(distilbert_errors)/len(test_labels)*100:.2f}%)")

print("\nSample Edge Cases Misclassified by DistilBERT:")
for _, row in distilbert_errors.head(5).iterrows():
    print(f"Query:     "{row['query']}"")
    print(f"True:      {row['true_intent']}")
    print(f"Predicted: {row['bert_pred']}")
    print("-" * 60)


SyntaxError: invalid syntax. Perhaps you forgot a comma? (3792642207.py, line 13)

## 8. Exporting Model Artifacts for Production

We serialize the fine-tuned DistilBERT weights, configuration, and tokenizer files into `models/distilbert_intent/` for our RAG and FastAPI backends.


In [9]:
output_paths = [
    Path("C:/Users/111196/Desktop/CHATBOT/models/distilbert_intent"),
    Path("../models/distilbert_intent"),
    Path("models/distilbert_intent")
]

save_dir = None
for p in output_paths:
    try:
        p.mkdir(parents=True, exist_ok=True)
        save_dir = p
        break
    except Exception:
        continue

if save_dir is None:
    save_dir = Path("models/distilbert_intent")
    save_dir.mkdir(parents=True, exist_ok=True)

# Save model and tokenizer
model.save_pretrained(save_dir)
tokenizer.save_pretrained(save_dir)

# Also save label mapping
joblib_dir = save_dir / "label_mapping.joblib"
import joblib
joblib.dump({"id2label": id2label, "label2id": label2id, "classes": label_encoder.classes_}, joblib_dir)

print(f"Successfully exported DistilBERT model artifacts to: {save_dir.resolve()}")
print("Exported files:", os.listdir(save_dir))


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Successfully exported DistilBERT model artifacts to: C:\Users\111196\Desktop\CHATBOT\models\distilbert_intent
Exported files: ['config.json', 'label_mapping.joblib', 'model.safetensors', 'tokenizer.json', 'tokenizer_config.json']


## 9. Error Analysis & Reflection (Your DS Work)

*Fill in your observations below:*

### 1. Where DistilBERT Outperformed Baseline TF-IDF:
- *(Find 2-3 specific customer queries that TF-IDF misclassified due to shared keywords like 'order' or 'cancel', but DistilBERT classified correctly due to syntax, word order, and attention).*

### 2. Genuinely Hard Ambiguities:
- *(Examine 2 queries that were misclassified even by DistilBERT. For example: does 'Where is my order refund' relate more to `track_order` or `check_refund_policy`? Explain the intrinsic semantic overlap).*

### 3. Latency vs. Accuracy Trade-off:
- *(Discuss the engineering trade-off: DistilBERT provides higher accuracy and contextual reasoning, while TF-IDF is orders of magnitude faster and lighter on CPU-only edge deployments).*
